## Transform Data For Gold Layer

In [1]:
df = spark.read.table("sales_silver")

display(df)


StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2437e6e-12fe-44be-9634-2ebd1c72b49f)

In [2]:
df_clean = df.na.drop()

display(df_clean)

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05658051-6764-4f61-8fca-185dca8e5576)

In [3]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("dimdate_gold") \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Day" , IntegerType()) \
    .addColumn("Month", IntegerType()) \
    .addColumn("Year", IntegerType()) \
    .addColumn("mmmyyy", StringType()) \
    .addColumn("yyyymm", StringType()) \
    .execute()

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 5, Finished, Available, Finished, False)

In [4]:
df_gold = spark.read.table("dimdate_gold")

display(df_gold)

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 374404a6-9e02-461d-a682-59db9a839485)

In [5]:
from pyspark.sql.functions import col, dayofmonth , month, year, date_format

dfdimDate_gold = df_clean.dropDuplicates(["OrderDate"]).select(col("OrderDate"), \
    dayofmonth("OrderDate").alias("Day"), \
    month("OrderDate").alias("Month"), \
    year("OrderDate").alias("Year"), \
    date_format(col("OrderDate"), "MMM-yyyy").alias("mmmyyyy"), \
    date_format(col("OrderDate"), "yyyyMM").alias("yyyymm"), \
).orderBy("OrderDate")

display(dfdimDate_gold.head(10))

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7f12831-b154-4e56-bf0f-f5f8d2f1b806)

In [7]:
deltaTable.toDF().printSchema()

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 9, Finished, Available, Finished, False)

root
 |-- OrderDate: date (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- mmmyyy: string (nullable = true)
 |-- yyyymm: string (nullable = true)



In [9]:
from delta.tables import *


deltaTable = DeltaTable.forName(spark, "dimdate_gold")

dfUpdates = dfdimDate_gold

deltaTable.alias('gold') \
.merge(
    dfUpdates.alias('updates'),
    'gold.OrderDate = updates.OrderDate'
) \
.whenMatchedUpdate( set =
{

}) \
.whenNotMatchedInsert(values =
{
 
   "OrderDate" : "updates.OrderDate",
   "Day" : "updates.Day",
   "Month" : "updates.Month",
   "Year" : "updates.Year",
   "mmmyyy" : "updates.mmmyyyy",
   "yyyymm" : "updates.yyyymm",
   
}) \
.execute()

StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("dimcustomer_gold") \
    .addColumn("OrderID", IntegerType()) \
    .addColumn("CustomerID", StringType()) \
    .execute()


StatementMeta(, 51b87c99-29bd-4a16-931b-c4bbdf584816, 12, Finished, Available, Finished, False)